[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-1/lab-1.2-tiled-matmul.ipynb)

# LAB·1.2 · Tiled matmul with a carried accumulator

**Hardware:** correctness anywhere; the timing half needs a TPU runtime.

The op that owns the MXU. The new idea over LAB·1.1: a grid dimension that *carries state*. The K dimension is a reduction, so the output block at (i, j) is revisited once per K step and accumulates. This revisit-and-accumulate pattern is the heart of every kernel in this track.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


## The kernel

The body is four lines of algorithm. Everything in the `pallas_call` below it (grid order, block shapes, index maps) is schedule, and each choice is worth interrogating: why is K the innermost grid dim, and what would happen to the accumulator if it were outermost?

In [ ]:
def matmul_kernel(a_ref, b_ref, o_ref):
    k = pl.program_id(2)
    @pl.when(k == 0)
    def _init():
        o_ref[...] = jnp.zeros_like(o_ref)
    o_ref[...] += jnp.dot(a_ref[...], b_ref[...], preferred_element_type=jnp.float32).astype(o_ref.dtype)

In [ ]:
def matmul(a, b, bm=128, bn=128, bk=128):
    m, k = a.shape
    _, n = b.shape
    return pl.pallas_call(
        matmul_kernel,
        grid=(m // bm, n // bn, k // bk),
        in_specs=[
            pl.BlockSpec((bm, bk), lambda i, j, kk: (i, kk)),
            pl.BlockSpec((bk, bn), lambda i, j, kk: (kk, j)),
        ],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j, kk: (i, j)),
        out_shape=jax.ShapeDtypeStruct((m, n), a.dtype),
        interpret=INTERP,
    )(a, b)

a = jax.random.normal(jax.random.key(0), (512, 512), jnp.float32)
b = jax.random.normal(jax.random.key(1), (512, 512), jnp.float32)
check("matmul 512", matmul(a, b), a @ b, tol=1e-3)

## Measure against XLA (TPU runtime)

The gate criterion for this stage: within 15% of `jnp.dot` at 4096³ bf16. Expect to lose on the first block sizes you try; the sweep is the lesson. Record every result with the chip name.

In [ ]:
def bench(fn, *args, reps=20):
    fn(*args).block_until_ready()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(*args).block_until_ready()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)) * 1e6

if ON_TPU:
    N = 4096
    ab = jax.random.normal(jax.random.key(0), (N, N), jnp.bfloat16)
    bb = jax.random.normal(jax.random.key(1), (N, N), jnp.bfloat16)
    xla_us = bench(jax.jit(lambda x, y: x @ y), ab, bb)
    print(f"XLA: {xla_us:.0f} us")
    for bm, bn, bk in [(128, 128, 128), (256, 256, 256), (512, 512, 512), (512, 1024, 512)]:
        us = bench(jax.jit(lambda x, y: matmul(x, y, bm, bn, bk)), ab, bb)
        print(f"pallas ({bm},{bn},{bk}): {us:8.0f} us  ratio {us / xla_us:5.2f}x  chip={jax.devices()[0].device_kind}")
else:
    print("Timing needs a TPU runtime; correctness above ran in interpret mode.")

## Exercise

Two questions to answer with the sweep, in a sentence each: which block dimension moves the needle most, and why does the winning config stop improving past a certain block size? (Both answers live in VMEM arithmetic, not folklore.)